In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Einen Job überwachen oder abbrechen

Sieh dir eine Liste deiner Workloads auf der [Workloads-Seite](https://quantum.cloud.ibm.com/workloads) an.

## Job-Status anzeigen
Gehe zu deiner [Workloads-Tabelle](https://quantum.cloud.ibm.com/workloads) und prüfe in der Spalte „Status", ob ein Job abgeschlossen wurde oder fehlgeschlagen ist.

## Verbleibendes Kontingent anzeigen
Gehe zu deiner [Instanzen-Tabelle](https://quantum.cloud.ibm.com/instances) und wähle den Tab aus, der dem Plan entspricht, für den du das verbleibende Kontingent anzeigen möchtest. Die insgesamt genutzte Zeit und die verbleibende Zeit in deinem Plan werden angezeigt.

## Metriken zur Anzahl der eingereichten Jobs und Workloads anzeigen
Gehe zur [Analytics-Seite](https://quantum.cloud.ibm.com/analytics), um die Gesamtzahl der eingereichten Jobs sowie die Anzahl der Batch-Workloads und Session-Workloads zu sehen. Beachte, dass du die Analytics-Seite nur für Konten anzeigen kannst, die du besitzt oder verwaltest.

## Einen Job überwachen
Verwende die Job-Instanz, um den Job-Status zu prüfen oder die Ergebnisse abzurufen, indem du den passenden Befehl aufrufst:

|                               |                                                                                                                                                                                                                                     |
| ----------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Sieh dir die Job-Ergebnisse unmittelbar nach Abschluss des Jobs an. Job-Ergebnisse sind verfügbar, sobald der Job abgeschlossen ist. Daher ist job.result() ein blockierender Aufruf, bis der Job abgeschlossen ist.                |
| job.job\_id()                 | Gibt die ID zurück, die diesen Job eindeutig identifiziert. Um Job-Ergebnisse zu einem späteren Zeitpunkt abzurufen, wird die Job-ID benötigt. Es wird daher empfohlen, die IDs von Jobs zu speichern, die du später abrufen möchtest. |
| job.status()                  | Prüfe den Job-Status.                                                                                                                                                                                                               |
| job = service.job(\<job\_id>) | Rufe einen zuvor eingereichten Job ab. Dieser Aufruf erfordert die Job-ID.                                                                                                                                                          |

<span id="retrieve-later"></span>
## Job-Ergebnisse zu einem späteren Zeitpunkt abrufen
Rufe `service.job(\<job\_id>)` auf, um einen zuvor eingereichten Job abzurufen. Wenn du die Job-ID nicht hast oder mehrere Jobs auf einmal abrufen möchtest – einschließlich Jobs von außer Betrieb genommenen QPUs (Quantenprozessoren) –, rufe stattdessen `service.jobs()` mit optionalen Filtern auf. Siehe [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` gibt auch Jobs zurück, die mit dem veralteten Paket `qiskit-ibm-provider` ausgeführt wurden. Jobs, die mit dem noch älteren (ebenfalls veralteten) Paket `qiskit-ibmq-provider` eingereicht wurden, sind nicht mehr verfügbar.

### Beispiel
Dieses Beispiel gibt die 10 neuesten Runtime-Jobs zurück, die auf `my_backend` ausgeführt wurden:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Einen Job abbrechen
Du kannst einen Job über das IBM Quantum Platform-Dashboard abbrechen, entweder auf der Workloads-Seite oder auf der Detailseite eines bestimmten Workloads. Klicke auf der Workloads-Seite auf das Überlaufmenü am Ende der Zeile des Workloads und wähle „Abbrechen" aus. Wenn du dich auf der Detailseite eines bestimmten Workloads befindest, verwende das Dropdown-Menü „Aktionen" oben auf der Seite und wähle „Abbrechen" aus.

In Qiskit verwendest du `job.cancel()`, um einen Job abzubrechen.

<span id="execution-spans"></span>
## Sampler-Ausführungsspannen anzeigen
Die Ergebnisse von [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)-Jobs, die in Qiskit Runtime ausgeführt werden, enthalten Informationen zur Ausführungszeit in ihren Metadaten.
Diese Zeitinformationen können verwendet werden, um obere und untere Zeitstempelgrenzen festzulegen, wann bestimmte Shots auf dem QPU ausgeführt wurden.
Shots werden in [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span)-Objekten gruppiert, die jeweils eine Startzeit, eine Endzeit und eine Angabe darüber enthalten, welche Shots in der Spanne gesammelt wurden.

Eine Ausführungsspanne legt fest, welche Daten während ihres Zeitfensters ausgeführt wurden, indem sie eine [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask)-Methode bereitstellt. Diese Methode gibt für einen beliebigen [Primitive Unified Block (PUB)](/guides/primitive-input-output)-Index eine boolesche Maske zurück, die für alle Shots, die während ihres Zeitfensters ausgeführt wurden, `True` ist. PUBs werden nach der Reihenfolge indiziert, in der sie dem Sampler-Aufruf übergeben wurden. Wenn ein PUB beispielsweise die Form `(2, 3)` hat und mit vier Shots ausgeführt wurde, hat die Maske die Form `(2, 3, 4)`. Vollständige Details findest du auf der API-Seite [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span).

Beispiel:
Um Informationen zu Ausführungsspannen anzuzeigen, sieh dir die Metadaten des von `SamplerV2` zurückgegebenen Ergebnisses an, das in Form eines `ExecutionSpans`-Objekts vorliegt. Dieses Objekt ist ein listenähnlicher Container, der Instanzen von Unterklassen von `ExecutionSpan` enthält, wie z. B. `SliceSpan`: